In [0]:
# COMMAND ----------
# 0. Setup

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

current_user = spark.sql("SELECT current_user()").collect()[0][0]

# Keep RAW_PATH unchanged
RAW_PATH = f"/Workspace/Users/{current_user}/raw_data"

CATALOG = "workspace"
SCHEMA = "medallion"

# Create schema under catalog workspace
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog     : {CATALOG}")
print(f"Schema      : {SCHEMA}")
print(f"Raw path    : {RAW_PATH}")

# COMMAND ----------
# 1. Bronze writer

def write_bronze(df, table_name, mode="overwrite"):
    """
    Write a DataFrame to a Bronze Delta table.

    Example output tables:
        workspace.medallion.bronze_product
        workspace.medallion.bronze_customer
    """

    target_table = f"{CATALOG}.{SCHEMA}.bronze_{table_name}"

    # Clean column names
    fixed_columns = {
        c: (
            c.strip()
             .replace(" ", "_")
             .replace(",", "_")
             .replace(";", "_")
             .replace("{", "_")
             .replace("}", "_")
             .replace("(", "_")
             .replace(")", "_")
             .replace("\n", "_")
             .replace("\t", "_")
             .replace("=", "_")
        )
        for c in df.columns
    }

    df_fixed = df.select([F.col(c).alias(fixed_columns[c]) for c in df.columns])

    df_bronze = (
        df_fixed
        .withColumn("_ingest_time", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_batch_date", F.lit(datetime.now().strftime("%Y-%m-%d")))
    )

    (
        df_bronze.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    count = spark.table(target_table).count()
    print(f"{target_table:45s} {count:>8,} rows")

    return count

# COMMAND ----------
# 2. Read RAW CSV

def read_raw_csv(filename, **kwargs):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")   # Bronze = everything STRING
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{RAW_PATH}/{filename}", **kwargs)
    )

# COMMAND ----------
# 3. Ingest RAW files → Bronze tables

# Product
df = read_raw_csv("RAW_Product.csv")
print("RAW_Product columns:", df.columns)
df.show(3, truncate=False)
write_bronze(df, "product")

# Product
df = read_raw_csv("RAW_Date.csv")
print("RAW_Date columns:", df.columns)
df.show(3, truncate=False)
write_bronze(df, "date")

# Customer
df = read_raw_csv("RAW_Customer.csv")
print("RAW_Customer columns:", df.columns)
df.show(3, truncate=False)
write_bronze(df, "customer")

# Reseller
df = read_raw_csv("RAW_Reseller.csv")
print("RAW_Reseller columns:", df.columns)
df.show(3, truncate=False)
write_bronze(df, "reseller")

# Sales Territory
df = read_raw_csv("RAW_SalesTerritory.csv")
print("RAW_SalesTerritory columns:", df.columns)
df.show(3, truncate=False)
write_bronze(df, "sales_territory")

# Sales Order
df = read_raw_csv("RAW_SalesOrder.csv")
print("RAW_SalesOrder columns:", df.columns)
df.show(3, truncate=False)
write_bronze(df, "sales_order")

#Sales
df = read_raw_csv("RAW_Sales.csv")
print("Columns:", df.columns)
df.show(3, truncate=False)
write_bronze(df, "sales")
# COMMAND ----------
# 4. Verify tables

spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 2 — Silver Layer: Cleaning + Data Modeling
# MAGIC
# MAGIC **What happens here:**
# MAGIC 1. Clean each Bronze table (nulls, types, dedup, casing)
# MAGIC 2. Generate surrogate keys for every dimension table
# MAGIC 3. Join the fact table (Sales) to all dimensions to replace natural keys with surrogate keys
# MAGIC 4. Output a proper Star Schema in Silver Delta tables
# MAGIC
# MAGIC ```
# MAGIC Bronze tables in workspace.medallion:
# MAGIC   bronze_customer
# MAGIC   bronze_product
# MAGIC   bronze_reseller
# MAGIC   bronze_sales_territory
# MAGIC   bronze_date
# MAGIC   bronze_sales_order
# MAGIC   bronze_sales
# MAGIC
# MAGIC   --> Silver tables in workspace.medallion:
# MAGIC       silver_dim_customer
# MAGIC       silver_dim_product
# MAGIC       silver_dim_reseller
# MAGIC       silver_dim_territory
# MAGIC       silver_dim_date
# MAGIC       silver_dim_salesorder
# MAGIC       silver_fact_sales
# MAGIC ```

# COMMAND ----------
# MAGIC %md ## 0. Configuration

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA  = "medallion"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")

# Fully qualified Bronze tables
BRONZE_CUSTOMER        = f"{CATALOG}.{SCHEMA}.bronze_customer"
BRONZE_PRODUCT         = f"{CATALOG}.{SCHEMA}.bronze_product"
BRONZE_RESELLER        = f"{CATALOG}.{SCHEMA}.bronze_reseller"
BRONZE_TERRITORY       = f"{CATALOG}.{SCHEMA}.bronze_sales_territory"
BRONZE_DATE            = f"{CATALOG}.{SCHEMA}.bronze_date"
BRONZE_SALES_ORDER     = f"{CATALOG}.{SCHEMA}.bronze_sales_order"
BRONZE_SALES           = f"{CATALOG}.{SCHEMA}.bronze_sales"

# Fully qualified Silver tables
SILVER_DIM_CUSTOMER    = f"{CATALOG}.{SCHEMA}.silver_dim_customer"
SILVER_DIM_PRODUCT     = f"{CATALOG}.{SCHEMA}.silver_dim_product"
SILVER_DIM_RESELLER    = f"{CATALOG}.{SCHEMA}.silver_dim_reseller"
SILVER_DIM_TERRITORY   = f"{CATALOG}.{SCHEMA}.silver_dim_territory"
SILVER_DIM_DATE        = f"{CATALOG}.{SCHEMA}.silver_dim_date"
SILVER_DIM_SALESORDER  = f"{CATALOG}.{SCHEMA}.silver_dim_salesorder"
SILVER_FACT_SALES      = f"{CATALOG}.{SCHEMA}.silver_fact_sales"

def write_silver(df, table_name, mode="overwrite"):
    """
    Save Silver table as:
      workspace.medallion.silver_<table_name>
    """
    target_table = f"{CATALOG}.{SCHEMA}.silver_{table_name}"

    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    n = spark.table(target_table).count()
    print(f"  {target_table:45s} {n:>8,} rows")
    return df

def drop_if_exists(df, cols):
    existing = [c for c in cols if c in df.columns]
    return df.drop(*existing) if existing else df

def rename_if_exists(df, old_name, new_name):
    return df.withColumnRenamed(old_name, new_name) if old_name in df.columns else df

def trim_all_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
    return df

def parse_date(col_name):
    c = f"`{col_name}`"
    return F.coalesce(
        F.to_date(F.expr(f"try_to_timestamp({c}, 'yyyy-MM-dd')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'MM/dd/yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'dd-MM-yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'yyyyMMdd')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'MMM dd, yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'yyyy-MM-dd HH:mm:ss')"))
    )

def clean_price(col_name):
    return F.regexp_replace(F.col(col_name), r"[^\d\.\-]", "").cast(DoubleType())


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 1. dim_customer

# COMMAND ----------

df = spark.table(BRONZE_CUSTOMER)

df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date", "_source"])

df = rename_if_exists(df, "Customer_ID", "CustomerNaturalKey")
df = rename_if_exists(df, "Customer", "CustomerName")
df = rename_if_exists(df, "State-Province", "StateProvince")
df = rename_if_exists(df, "State_Province", "StateProvince")
df = rename_if_exists(df, "Country-Region", "CountryRegion")
df = rename_if_exists(df, "Country_Region", "CountryRegion")
df = rename_if_exists(df, "Postal Code", "PostalCode")
df = rename_if_exists(df, "Postal_Code", "PostalCode")
df = rename_if_exists(df, "Customer_ID", "CustomerNaturalKey")
df = rename_if_exists(df, "Customer_ID_", "CustomerNaturalKey")
df = rename_if_exists(df, "Customer_ID__", "CustomerNaturalKey")

# Bronze column cleaning may already have replaced spaces with underscores
df = rename_if_exists(df, "Customer_ID", "CustomerNaturalKey")
df = rename_if_exists(df, "Customer", "CustomerName")

df = trim_all_string_columns(df)

if "StateProvince" in df.columns:
    df = df.withColumn("StateProvince", F.initcap(F.col("StateProvince")))
if "CountryRegion" in df.columns:
    df = df.withColumn("CountryRegion", F.initcap(F.col("CountryRegion")))

if "City" in df.columns:
    df = df.withColumn(
        "City",
        F.when(F.col("City").isNull() | (F.col("City") == ""), F.lit("Unknown")).otherwise(F.col("City"))
    )

if "PostalCode" in df.columns:
    df = df.withColumn(
        "PostalCode",
        F.when(F.col("PostalCode").isNull() | (F.col("PostalCode") == ""), F.lit("00000")).otherwise(F.col("PostalCode"))
    )

if "CustomerName" in df.columns:
    df = df.withColumn(
        "CustomerName",
        F.when(F.col("CustomerName").isNull() | (F.col("CustomerName") == ""), F.lit("Unknown")).otherwise(F.col("CustomerName"))
    )

df = df.dropDuplicates(["CustomerNaturalKey"])

w = Window.orderBy("CustomerNaturalKey")
df = df.withColumn("CustomerKey", F.row_number().over(w))

df = df.select(
    "CustomerKey",
    "CustomerNaturalKey",
    "CustomerName",
    "City",
    "StateProvince",
    "CountryRegion",
    "PostalCode"
)

df.show(5, truncate=False)
write_silver(df, "dim_customer")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 2. dim_product

# COMMAND ----------

df = spark.table(BRONZE_PRODUCT)
df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date"])

df = rename_if_exists(df, "Standard_Cost", "StandardCost")
df = rename_if_exists(df, "List_Price", "ListPrice")

df = trim_all_string_columns(df)

df = df.withColumn("ListPrice", clean_price("ListPrice"))
df = df.withColumn("StandardCost", clean_price("StandardCost"))

df = df.filter(F.col("ListPrice") > 0)

if "Color" in df.columns:
    df = df.withColumn(
        "Color",
        F.when(F.col("Color").isNull() | (F.col("Color") == ""), F.lit("No Color")).otherwise(F.col("Color"))
    )

if "Category" in df.columns:
    df = df.withColumn(
        "Category",
        F.when(F.col("Category").isNull() | (F.col("Category") == ""), F.lit("Unknown")).otherwise(F.col("Category"))
    )

df = df.withColumn(
    "StandardCost",
    F.when(F.col("StandardCost").isNull(), F.lit(0.0)).otherwise(F.col("StandardCost"))
)

if "Category" in df.columns:
    df = df.withColumn("Category", F.initcap(F.col("Category")))
if "Subcategory" in df.columns:
    df = df.withColumn("Subcategory", F.initcap(F.col("Subcategory")))
if "Color" in df.columns:
    df = df.withColumn("Color", F.initcap(F.col("Color")))

df = df.dropDuplicates(["SKU"])

w = Window.orderBy("SKU")
df = df.withColumn("ProductKey", F.row_number().over(w))

df = df.select(
    "ProductKey",
    "SKU",
    "Product",
    "Model",
    "Category",
    "Subcategory",
    "Color",
    "ListPrice",
    "StandardCost"
)

df.show(5, truncate=False)
write_silver(df, "dim_product")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 3. dim_reseller

# COMMAND ----------

df = spark.table(BRONZE_RESELLER)
df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date"])

df = rename_if_exists(df, "Reseller_ID", "ResellerNaturalKey")
df = rename_if_exists(df, "Business_Type", "BusinessType")
df = rename_if_exists(df, "State-Province", "StateProvince")
df = rename_if_exists(df, "State_Province", "StateProvince")
df = rename_if_exists(df, "Country-Region", "CountryRegion")
df = rename_if_exists(df, "Country_Region", "CountryRegion")
df = rename_if_exists(df, "Postal_Code", "PostalCode")

df = trim_all_string_columns(df)

if "BusinessType" in df.columns:
    df = df.withColumn("BusinessType", F.initcap(F.col("BusinessType")))
if "CountryRegion" in df.columns:
    df = df.withColumn("CountryRegion", F.initcap(F.col("CountryRegion")))
if "StateProvince" in df.columns:
    df = df.withColumn("StateProvince", F.initcap(F.col("StateProvince")))

if "BusinessType" in df.columns:
    df = df.withColumn(
        "BusinessType",
        F.when(F.col("BusinessType").isNull() | (F.col("BusinessType") == ""), F.lit("Unknown")).otherwise(F.col("BusinessType"))
    )

if "City" in df.columns:
    df = df.withColumn(
        "City",
        F.when(F.col("City").isNull() | (F.col("City") == ""), F.lit("Unknown")).otherwise(F.col("City"))
    )

if "PostalCode" in df.columns:
    df = df.withColumn(
        "PostalCode",
        F.when(F.col("PostalCode").isNull() | (F.col("PostalCode") == ""), F.lit("00000")).otherwise(F.col("PostalCode"))
    )

df = df.dropDuplicates(["ResellerNaturalKey"])

w = Window.orderBy("ResellerNaturalKey")
df = df.withColumn("ResellerKey", F.row_number().over(w))

df = df.select(
    "ResellerKey",
    "ResellerNaturalKey",
    "Reseller",
    "BusinessType",
    "City",
    "StateProvince",
    "CountryRegion",
    "PostalCode"
)

df.show(5, truncate=False)
write_silver(df, "dim_reseller")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 4. dim_territory

# COMMAND ----------

df = spark.table(BRONZE_TERRITORY)
df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date"])

# Optional junk column
df = drop_if_exists(df, ["notes", "Notes"])

df = rename_if_exists(df, "REGION", "Region")
df = rename_if_exists(df, "Sales_Group", "SalesGroup")
df = rename_if_exists(df, "Sales_Group_", "SalesGroup")

for c in ["Region", "Country", "SalesGroup"]:
    if c in df.columns:
        df = df.withColumn(c, F.initcap(F.trim(F.col(c))))

df = df.dropDuplicates(["Region", "Country"])

w = Window.orderBy("Region")
df = df.withColumn("SalesTerritoryKey", F.row_number().over(w))

df = df.select("SalesTerritoryKey", "Region", "Country", "SalesGroup")

df.show(truncate=False)
write_silver(df, "dim_territory")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 5. dim_date

# COMMAND ----------

df = spark.table(BRONZE_DATE)
df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date"])

# Bronze columns likely have underscores
df = rename_if_exists(df, "Full_Date", "FullDate")
df = rename_if_exists(df, "Fiscal_Year", "FiscalYear")
df = rename_if_exists(df, "Fiscal_Quarter", "FiscalQuarter")

df = df.withColumn("Date", parse_date("Date"))

if "FullDate" in df.columns:
    df = df.withColumn("FullDate", parse_date("FullDate"))

if "Month" in df.columns:
    # Preserve original Month string if it is not a real date
    df = df.withColumn("MonthDateParsed", parse_date("Month"))

if "FiscalYear" in df.columns:
    df = df.withColumn(
        "FiscalYear",
        F.concat(F.lit("FY"), F.regexp_extract(F.col("FiscalYear"), r"(\d{4})", 1))
    )

fiscal_month = F.when(F.month("Date") >= 7, F.month("Date") - 6).otherwise(F.month("Date") + 6)
fiscal_quarter_num = F.ceil(fiscal_month / 3).cast(IntegerType())

if "FiscalQuarter" in df.columns and "FiscalYear" in df.columns:
    df = df.withColumn(
        "FiscalQuarter",
        F.when(
            F.col("FiscalQuarter").isNull() | (F.col("FiscalQuarter") == ""),
            F.concat(F.col("FiscalYear"), F.lit(" Q"), fiscal_quarter_num.cast(StringType()))
        ).otherwise(F.col("FiscalQuarter"))
    )

df = df.withColumn("DateKey", F.date_format(F.col("Date"), "yyyyMMdd").cast(IntegerType()))
df = df.withColumn("MonthKey", F.date_format(F.col("Date"), "yyyyMM").cast(IntegerType()))

df = df.filter(F.col("Date").isNotNull()).dropDuplicates(["DateKey"])

select_cols = ["DateKey", "Date"]
if "FiscalYear" in df.columns:
    select_cols.append("FiscalYear")
if "FiscalQuarter" in df.columns:
    select_cols.append("FiscalQuarter")
if "Month" in df.columns:
    select_cols.append("Month")
if "FullDate" in df.columns:
    select_cols.append("FullDate")
select_cols.append("MonthKey")

df = df.select(*select_cols)

df.show(10, truncate=False)
write_silver(df, "dim_date")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 6. dim_salesorder

# COMMAND ----------

df = spark.table(BRONZE_SALES_ORDER)
df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date", "created_date", "Created_Date"])

df = rename_if_exists(df, "Sales_Order", "SalesOrder")
df = rename_if_exists(df, "Sales_Order_Line", "SalesOrderLine")

if "Channel" in df.columns:
    df = df.withColumn("Channel", F.initcap(F.trim(F.col("Channel"))))
if "SalesOrder" in df.columns:
    df = df.withColumn("SalesOrder", F.trim(F.col("SalesOrder")))
if "SalesOrderLine" in df.columns:
    df = df.withColumn("SalesOrderLine", F.trim(F.col("SalesOrderLine")))

if "Channel" in df.columns:
    df = df.withColumn(
        "Channel",
        F.when(F.col("Channel").isNull() | (F.col("Channel") == ""), F.lit("Unknown")).otherwise(F.col("Channel"))
    )

df = df.dropDuplicates(["SalesOrderLine"])

df = (
    df.withColumn("_order_num", F.regexp_extract(F.col("SalesOrder"), r"SO(\d+)", 1).cast(LongType()))
      .withColumn("_line_num", F.regexp_extract(F.col("SalesOrderLine"), r"-\s*(\d+)$", 1).cast(LongType()))
      .withColumn("SalesOrderLineKey", (F.col("_order_num") * 100 + F.col("_line_num")).cast(LongType()))
      .drop("_order_num", "_line_num")
)

df = df.select("SalesOrderLineKey", "SalesOrder", "SalesOrderLine", "Channel")

df.show(5, truncate=False)
write_silver(df, "dim_salesorder")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 7. fact_sales

# COMMAND ----------

df_sales = spark.table(BRONZE_SALES)
df_sales = drop_if_exists(df_sales, ["_ingest_time", "_source_file", "_batch_date", "_batch_id"])

# Normalize likely Bronze column names
rename_map = {
    "Customer_ID": "CustomerID",
    "Reseller_ID": "ResellerID",
    "Sales_Order_Line": "SalesOrderLine",
    "Sales_Order": "SalesOrder",
    "Order_Date": "OrderDate",
    "Due_Date": "DueDate",
    "Ship_Date": "ShipDate",
    "Order_Quantity": "OrderQuantity",
    "Unit_Price": "UnitPrice",
    "Unit_Price_Discount_Pct": "UnitPriceDiscountPct",
    "Product_Standard_Cost": "ProductStandardCost",
    "Total_Product_Cost": "TotalProductCost",
    "Extended_Amount": "ExtendedAmount",
    "Sales_Amount": "SalesAmount"
}

for old_name, new_name in rename_map.items():
    df_sales = rename_if_exists(df_sales, old_name, new_name)

if "Channel" in df_sales.columns:
    df_sales = df_sales.withColumn("Channel", F.initcap(F.trim(F.col("Channel"))))
if "Region" in df_sales.columns:
    df_sales = df_sales.withColumn("Region", F.initcap(F.trim(F.col("Region"))))
if "Country" in df_sales.columns:
    df_sales = df_sales.withColumn("Country", F.initcap(F.trim(F.col("Country"))))

if "OrderDate" in df_sales.columns:
    df_sales = df_sales.withColumn("OrderDate", parse_date("OrderDate"))
if "DueDate" in df_sales.columns:
    df_sales = df_sales.withColumn("DueDate", parse_date("DueDate"))
if "ShipDate" in df_sales.columns:
    df_sales = df_sales.withColumn("ShipDate", parse_date("ShipDate"))

if "UnitPriceDiscountPct" in df_sales.columns:
    df_sales = df_sales.withColumn(
        "UnitPriceDiscountPct",
        F.when(
            F.col("UnitPriceDiscountPct").cast(StringType()).endswith("%"),
            F.regexp_extract(F.col("UnitPriceDiscountPct"), r"([\d\.]+)%", 1).cast(DoubleType()) / 100
        ).otherwise(F.col("UnitPriceDiscountPct").cast(DoubleType()))
    )

numeric_casts = {
    "OrderQuantity": IntegerType(),
    "UnitPrice": DoubleType(),
    "ProductStandardCost": DoubleType(),
    "TotalProductCost": DoubleType(),
    "ExtendedAmount": DoubleType(),
    "SalesAmount": DoubleType()
}

for c, dtype in numeric_casts.items():
    if c in df_sales.columns:
        if c == "OrderQuantity":
            df_sales = df_sales.withColumn(c, F.round(F.col(c).cast(DoubleType()), 0).cast(dtype))
        else:
            df_sales = df_sales.withColumn(c, F.col(c).cast(dtype))

df_sales = df_sales.withColumn(
    "IsReturn",
    F.when(F.col("SalesAmount") < 0, F.lit(1)).otherwise(F.lit(0))
)

df_sales = df_sales.dropDuplicates(["SalesOrderLine"])

dim_cust = spark.table(SILVER_DIM_CUSTOMER).select("CustomerKey", "CustomerNaturalKey")
dim_prod = spark.table(SILVER_DIM_PRODUCT).select("ProductKey", "SKU")
dim_res  = spark.table(SILVER_DIM_RESELLER).select("ResellerKey", "ResellerNaturalKey")
dim_terr = spark.table(SILVER_DIM_TERRITORY).select("SalesTerritoryKey", "Region", "Country")
dim_date = spark.table(SILVER_DIM_DATE).select("DateKey", "Date")
dim_so   = spark.table(SILVER_DIM_SALESORDER).select("SalesOrderLineKey", "SalesOrderLine")

df_fact = (
    df_sales
    .join(
        dim_cust.withColumnRenamed("CustomerNaturalKey", "CustomerID"),
        on="CustomerID",
        how="left"
    )
    .join(dim_prod, on="SKU", how="left")
    .join(
        dim_res.withColumnRenamed("ResellerNaturalKey", "ResellerID"),
        on="ResellerID",
        how="left"
    )
    .join(dim_terr, on=["Region", "Country"], how="left")
    .join(
        dim_date.withColumnRenamed("Date", "_dim_date"),
        F.col("OrderDate") == F.col("_dim_date"),
        how="left"
    )
    .drop("_dim_date")
    .join(
        dim_so.withColumnRenamed("SalesOrderLine", "SalesOrderLine"),
        on="SalesOrderLine",
        how="left"
    )
)

df_fact = df_fact.select(
    "SalesOrderLineKey",
    "CustomerKey",
    "ProductKey",
    "ResellerKey",
    "SalesTerritoryKey",
    F.col("DateKey").alias("OrderDateKey"),
    "Channel",
    "OrderQuantity",
    "UnitPrice",
    "UnitPriceDiscountPct",
    "ProductStandardCost",
    "TotalProductCost",
    "ExtendedAmount",
    "SalesAmount",
    "OrderDate",
    "DueDate",
    "ShipDate",
    "IsReturn"
)

print("=== Join Quality Check ===")
print(f"Total fact rows         : {df_fact.count():,}")
print(f"NULL CustomerKey        : {df_fact.filter(F.col('CustomerKey').isNull()).count():,}")
print(f"NULL ProductKey         : {df_fact.filter(F.col('ProductKey').isNull()).count():,}")
print(f"NULL SalesTerritoryKey  : {df_fact.filter(F.col('SalesTerritoryKey').isNull()).count():,}")
print(f"NULL OrderDateKey       : {df_fact.filter(F.col('OrderDateKey').isNull()).count():,}")
print(f"NULL SalesOrderLineKey  : {df_fact.filter(F.col('SalesOrderLineKey').isNull()).count():,}")
print(f"Return rows (IsReturn=1): {df_fact.filter(F.col('IsReturn') == 1).count():,}")

write_silver(df_fact, "fact_sales")


# COMMAND ----------
# MAGIC %md ## 8. Verify Star Schema

# COMMAND ----------

spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").show(truncate=False)

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT 'silver_dim_customer'   AS table_name, COUNT(*) AS rows FROM workspace.medallion.silver_dim_customer
# MAGIC UNION ALL SELECT 'silver_dim_product',      COUNT(*) FROM workspace.medallion.silver_dim_product
# MAGIC UNION ALL SELECT 'silver_dim_reseller',     COUNT(*) FROM workspace.medallion.silver_dim_reseller
# MAGIC UNION ALL SELECT 'silver_dim_territory',    COUNT(*) FROM workspace.medallion.silver_dim_territory
# MAGIC UNION ALL SELECT 'silver_dim_date',         COUNT(*) FROM workspace.medallion.silver_dim_date
# MAGIC UNION ALL SELECT 'silver_dim_salesorder',   COUNT(*) FROM workspace.medallion.silver_dim_salesorder
# MAGIC UNION ALL SELECT 'silver_fact_sales',       COUNT(*) FROM workspace.medallion.silver_fact_sales
# MAGIC ORDER BY table_name

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT
# MAGIC     d.FiscalYear,
# MAGIC     t.Region,
# MAGIC     t.Country,
# MAGIC     p.Category,
# MAGIC     f.Channel,
# MAGIC     COUNT(f.SalesOrderLineKey)   AS OrderLines,
# MAGIC     SUM(f.OrderQuantity)         AS TotalUnits,
# MAGIC     ROUND(SUM(f.SalesAmount), 2) AS TotalRevenue
# MAGIC FROM workspace.medallion.silver_fact_sales f
# MAGIC JOIN workspace.medallion.silver_dim_date d
# MAGIC   ON f.OrderDateKey = d.DateKey
# MAGIC JOIN workspace.medallion.silver_dim_territory t
# MAGIC   ON f.SalesTerritoryKey = t.SalesTerritoryKey
# MAGIC JOIN workspace.medallion.silver_dim_product p
# MAGIC   ON f.ProductKey = p.ProductKey
# MAGIC WHERE f.IsReturn = 0
# MAGIC GROUP BY d.FiscalYear, t.Region, t.Country, p.Category, f.Channel
# MAGIC ORDER BY TotalRevenue DESC
# MAGIC LIMIT 20

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 3 — Gold Layer: Business Aggregations
# MAGIC
# MAGIC Build analytics-ready tables directly from the Silver star schema.
# MAGIC These tables are what BI tools (Power BI, Tableau, Looker) connect to.
# MAGIC
# MAGIC | Gold Table | Business Question |
# MAGIC |------------|-------------------|
# MAGIC | `gold_sales_by_month`   | How much revenue per month/region/category? |
# MAGIC | `gold_product_ranking`  | Which products/categories sell most? |
# MAGIC | `gold_customer_summary` | Who are our best customers? |
# MAGIC | `gold_channel_compare`  | Reseller vs Internet: which channel performs better? |

# COMMAND ----------
# MAGIC %md ## 0. Configuration

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
SCHEMA  = "medallion"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")

# Silver tables
SILVER_FACT_SALES     = f"{CATALOG}.{SCHEMA}.silver_fact_sales"
SILVER_DIM_DATE       = f"{CATALOG}.{SCHEMA}.silver_dim_date"
SILVER_DIM_PRODUCT    = f"{CATALOG}.{SCHEMA}.silver_dim_product"
SILVER_DIM_CUSTOMER   = f"{CATALOG}.{SCHEMA}.silver_dim_customer"
SILVER_DIM_RESELLER   = f"{CATALOG}.{SCHEMA}.silver_dim_reseller"
SILVER_DIM_TERRITORY  = f"{CATALOG}.{SCHEMA}.silver_dim_territory"

# Gold tables
GOLD_SALES_BY_MONTH   = f"{CATALOG}.{SCHEMA}.gold_sales_by_month"
GOLD_PRODUCT_RANKING  = f"{CATALOG}.{SCHEMA}.gold_product_ranking"
GOLD_CUSTOMER_SUMMARY = f"{CATALOG}.{SCHEMA}.gold_customer_summary"
GOLD_CHANNEL_COMPARE  = f"{CATALOG}.{SCHEMA}.gold_channel_compare"

# Load Silver tables
fact   = spark.table(SILVER_FACT_SALES)
d_date = spark.table(SILVER_DIM_DATE)
d_prod = spark.table(SILVER_DIM_PRODUCT)
d_cust = spark.table(SILVER_DIM_CUSTOMER)
d_res  = spark.table(SILVER_DIM_RESELLER)
d_terr = spark.table(SILVER_DIM_TERRITORY)

def write_gold(df, table_name, mode="overwrite"):
    """
    Save Gold table as:
      workspace.medallion.gold_<table_name>
    """
    target_table = f"{CATALOG}.{SCHEMA}.gold_{table_name}"

    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    n = spark.table(target_table).count()
    print(f"  {target_table:45s} {n:>8,} rows")
    return df


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 1. gold_sales_by_month
# MAGIC Monthly revenue broken down by FiscalYear, FiscalQuarter, Region, Category, and Channel.
# MAGIC Includes month-over-month growth rate.

# COMMAND ----------

df = (
    fact
    .filter(F.col("IsReturn") == 0)
    .join(
        d_date.select("DateKey", "FiscalYear", "FiscalQuarter", "MonthKey"),
        fact.OrderDateKey == d_date.DateKey,
        "inner"
    )
    .join(
        d_terr.select("SalesTerritoryKey", "Region", "Country", "SalesGroup"),
        "SalesTerritoryKey",
        "left"
    )
    .join(
        d_prod.select("ProductKey", "Category", "Subcategory"),
        "ProductKey",
        "left"
    )
    .groupBy(
        "FiscalYear", "FiscalQuarter", "MonthKey",
        "Region", "Country", "SalesGroup",
        "Category", "Channel"
    )
    .agg(
        F.count("SalesOrderLineKey").alias("OrderLines"),
        F.sum("OrderQuantity").alias("TotalUnits"),
        F.round(F.sum("SalesAmount"), 2).alias("TotalRevenue"),
        F.round(F.avg("SalesAmount"), 2).alias("AvgOrderValue"),
        F.round(F.sum("TotalProductCost"), 2).alias("TotalCost"),
        F.round(F.sum("SalesAmount") - F.sum("TotalProductCost"), 2).alias("GrossProfit"),
        F.countDistinct("CustomerKey").alias("UniqueCustomers"),
    )
    .withColumn(
        "GrossMarginPct",
        F.when(
            F.col("TotalRevenue") != 0,
            F.round(F.col("GrossProfit") / F.col("TotalRevenue") * 100, 1)
        )
    )
    .withColumn(
        "PrevMonthRevenue",
        F.lag("TotalRevenue", 1).over(
            Window.partitionBy("Region", "Category", "Channel").orderBy("MonthKey")
        )
    )
    .withColumn(
        "MoMGrowthPct",
        F.when(
            F.col("PrevMonthRevenue").isNotNull() & (F.col("PrevMonthRevenue") != 0),
            F.round(
                (F.col("TotalRevenue") - F.col("PrevMonthRevenue"))
                / F.col("PrevMonthRevenue") * 100,
                1
            )
        )
    )
    .drop("PrevMonthRevenue")
    .withColumn("_gold_timestamp", F.current_timestamp())
)

print("Preview:")
df.orderBy("MonthKey", "Region").show(10, truncate=False)
write_gold(df, "sales_by_month")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 2. gold_product_ranking
# MAGIC Revenue and unit ranking per product, with % contribution to total revenue.

# COMMAND ----------

total_revenue = (
    fact
    .filter(F.col("IsReturn") == 0)
    .agg(F.sum("SalesAmount").alias("total_revenue"))
    .collect()[0]["total_revenue"]
)

df = (
    fact
    .filter(F.col("IsReturn") == 0)
    .join(
        d_prod.select("ProductKey", "SKU", "Product", "Model", "Category", "Subcategory", "Color"),
        "ProductKey",
        "left"
    )
    .groupBy("ProductKey", "SKU", "Product", "Model", "Category", "Subcategory", "Color")
    .agg(
        F.sum("OrderQuantity").alias("UnitsSold"),
        F.round(F.sum("SalesAmount"), 2).alias("TotalRevenue"),
        F.round(F.sum("TotalProductCost"), 2).alias("TotalCost"),
        F.round(F.avg("UnitPrice"), 2).alias("AvgSellingPrice"),
        F.count("SalesOrderLineKey").alias("OrderLines"),
        F.countDistinct("CustomerKey").alias("UniqueCustomers"),
    )
    .withColumn("GrossProfit", F.round(F.col("TotalRevenue") - F.col("TotalCost"), 2))
    .withColumn(
        "GrossMarginPct",
        F.when(
            F.col("TotalRevenue") != 0,
            F.round(F.col("GrossProfit") / F.col("TotalRevenue") * 100, 1)
        )
    )
    .withColumn(
        "RevSharePct",
        F.when(
            F.lit(total_revenue).isNotNull() & (F.lit(total_revenue) != 0),
            F.round(F.col("TotalRevenue") / F.lit(total_revenue) * 100, 2)
        )
    )
    .withColumn("RevenueRank", F.rank().over(Window.orderBy(F.desc("TotalRevenue"))))
    .withColumn(
        "RevenueRankInCategory",
        F.rank().over(Window.partitionBy("Category").orderBy(F.desc("TotalRevenue")))
    )
    .withColumn("_gold_timestamp", F.current_timestamp())
    .orderBy("RevenueRank")
)

print("Top 10 products by revenue:")
df.select(
    "RevenueRank", "Product", "Category", "TotalRevenue",
    "UnitsSold", "GrossMarginPct", "RevSharePct"
).show(10, truncate=False)

write_gold(df, "product_ranking")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 3. gold_customer_summary
# MAGIC Customer lifetime value + RFM segmentation (Recency, Frequency, Monetary).

# COMMAND ----------

df_rfm = (
    fact
    .filter(F.col("IsReturn") == 0)
    .filter(F.col("CustomerKey").isNotNull())
    .groupBy("CustomerKey")
    .agg(
        F.datediff(F.current_date(), F.max("OrderDate")).alias("RecencyDays"),
        F.count("SalesOrderLineKey").alias("Frequency"),
        F.round(F.sum("SalesAmount"), 2).alias("LifetimeValue"),
        F.round(F.avg("SalesAmount"), 2).alias("AvgOrderValue"),
        F.round(F.sum("TotalProductCost"), 2).alias("TotalCost"),
        F.max("OrderDate").alias("LastOrderDate"),
        F.min("OrderDate").alias("FirstOrderDate"),
        F.countDistinct("ProductKey").alias("ProductVariety"),
    )
)

df_rfm = (
    df_rfm
    .withColumn(
        "R_Score",
        F.when(F.col("RecencyDays") <= 30, 5)
         .when(F.col("RecencyDays") <= 90, 4)
         .when(F.col("RecencyDays") <= 180, 3)
         .when(F.col("RecencyDays") <= 365, 2)
         .otherwise(1)
    )
    .withColumn(
        "F_Score",
        F.when(F.col("Frequency") >= 20, 5)
         .when(F.col("Frequency") >= 10, 4)
         .when(F.col("Frequency") >= 5, 3)
         .when(F.col("Frequency") >= 2, 2)
         .otherwise(1)
    )
    .withColumn(
        "M_Score",
        F.when(F.col("LifetimeValue") >= 20000, 5)
         .when(F.col("LifetimeValue") >= 10000, 4)
         .when(F.col("LifetimeValue") >= 3000, 3)
         .when(F.col("LifetimeValue") >= 500, 2)
         .otherwise(1)
    )
    .withColumn("RFM_Score", F.col("R_Score") + F.col("F_Score") + F.col("M_Score"))
    .withColumn(
        "CustomerSegment",
        F.when(F.col("RFM_Score") >= 13, "Champions")
         .when(F.col("RFM_Score") >= 10, "Loyal")
         .when(F.col("RFM_Score") >= 7, "Potential")
         .when(F.col("RFM_Score") >= 5, "At Risk")
         .otherwise("Lost")
    )
    .withColumn("GrossProfit", F.round(F.col("LifetimeValue") - F.col("TotalCost"), 2))
)

df = (
    d_cust
    .select(
        "CustomerKey", "CustomerNaturalKey", "CustomerName",
        "City", "StateProvince", "CountryRegion"
    )
    .join(df_rfm, "CustomerKey", "left")
    .withColumn("Frequency", F.coalesce(F.col("Frequency"), F.lit(0)))
    .withColumn("LifetimeValue", F.coalesce(F.col("LifetimeValue"), F.lit(0.0)))
    .withColumn("CustomerSegment", F.coalesce(F.col("CustomerSegment"), F.lit("Lost")))
    .withColumn("_gold_timestamp", F.current_timestamp())
)

print("Segment distribution:")
df.groupBy("CustomerSegment").count().orderBy(F.desc("count")).show(truncate=False)
write_gold(df, "customer_summary")


# COMMAND ----------
# MAGIC %md
# MAGIC ---
# MAGIC ## 4. gold_channel_compare
# MAGIC Side-by-side: Internet (B2C) vs Reseller (B2B) performance by fiscal year.

# COMMAND ----------

df = (
    fact
    .filter(F.col("IsReturn") == 0)
    .join(
        d_date.select("DateKey", "FiscalYear"),
        fact.OrderDateKey == d_date.DateKey,
        "inner"
    )
    .join(
        d_prod.select("ProductKey", "Category"),
        "ProductKey",
        "left"
    )
    .groupBy("FiscalYear", "Channel", "Category")
    .agg(
        F.count("SalesOrderLineKey").alias("OrderLines"),
        F.sum("OrderQuantity").alias("UnitsSold"),
        F.round(F.sum("SalesAmount"), 2).alias("TotalRevenue"),
        F.round(F.avg("SalesAmount"), 2).alias("AvgOrderValue"),
        F.round(F.sum("TotalProductCost"), 2).alias("TotalCost"),
        F.countDistinct("CustomerKey").alias("UniqueCustomers"),
    )
    .withColumn("GrossProfit", F.round(F.col("TotalRevenue") - F.col("TotalCost"), 2))
    .withColumn(
        "GrossMarginPct",
        F.when(
            F.col("TotalRevenue") != 0,
            F.round(F.col("GrossProfit") / F.col("TotalRevenue") * 100, 1)
        )
    )
    .withColumn(
        "RevenuePerCustomer",
        F.when(
            F.col("UniqueCustomers") != 0,
            F.round(F.col("TotalRevenue") / F.col("UniqueCustomers"), 2)
        )
    )
    .withColumn("_gold_timestamp", F.current_timestamp())
    .orderBy("FiscalYear", "Channel", "Category")
)

print("Channel comparison:")
df.show(20, truncate=False)
write_gold(df, "channel_compare")


# COMMAND ----------
# MAGIC %md ## 5. Final Summary

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT 'gold_sales_by_month' AS gold_table, COUNT(*) AS rows
# MAGIC FROM workspace.medallion.gold_sales_by_month
# MAGIC UNION ALL
# MAGIC SELECT 'gold_product_ranking', COUNT(*)
# MAGIC FROM workspace.medallion.gold_product_ranking
# MAGIC UNION ALL
# MAGIC SELECT 'gold_customer_summary', COUNT(*)
# MAGIC FROM workspace.medallion.gold_customer_summary
# MAGIC UNION ALL
# MAGIC SELECT 'gold_channel_compare', COUNT(*)
# MAGIC FROM workspace.medallion.gold_channel_compare

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT
# MAGIC     Region,
# MAGIC     Country,
# MAGIC     ROUND(SUM(TotalRevenue), 0) AS Revenue,
# MAGIC     SUM(OrderLines)             AS Orders
# MAGIC FROM workspace.medallion.gold_sales_by_month
# MAGIC GROUP BY Region, Country
# MAGIC ORDER BY Revenue DESC
# MAGIC LIMIT 5

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT
# MAGIC     Channel,
# MAGIC     ROUND(SUM(TotalRevenue), 0) AS TotalRevenue,
# MAGIC     SUM(UniqueCustomers)        AS Customers,
# MAGIC     ROUND(AVG(GrossMarginPct), 1) AS AvgMarginPct
# MAGIC FROM workspace.medallion.gold_channel_compare
# MAGIC GROUP BY Channel
# MAGIC ORDER BY TotalRevenue DESC

# COMMAND ----------
# MAGIC %md
# MAGIC ## ✅ Gold Complete
# MAGIC
# MAGIC | Table | Purpose |
# MAGIC |-------|---------|
# MAGIC | `gold_sales_by_month`   | Trend analysis, MoM growth, margin by region/category |
# MAGIC | `gold_product_ranking`  | Product performance, revenue share, margin ranking |
# MAGIC | `gold_customer_summary` | RFM segmentation, lifetime value, customer loyalty |
# MAGIC | `gold_channel_compare`  | Internet vs Reseller comparison by year and category |
# MAGIC
# MAGIC Connect any of these tables directly to **Power BI / Tableau**.